# 05 — Evaluation

Evaluate the trained YOLOv11 checkpoint on the held-out **test** split: precision/recall, mAP@0.5, mAP@0.5:0.95, confusion matrix, per-class PR curves, model-variant comparison, and a qualitative prediction grid.

In [ ]:
# Install dependencies
%pip install -q "ultralytics==8.3.*" pandas numpy matplotlib seaborn pillow
# On a bare Linux box (no system libGL) OpenCV import can fail — if so, uncomment:
# !apt-get update -qq && apt-get install -y -qq libgl1

In [ ]:
# Imports
from ultralytics import YOLO
from pathlib import Path
import random
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

In [ ]:
# Config — paths, classes, eval thresholds
DATA_YAML    = Path('../data/dataset/data.yaml').resolve()
WEIGHTS      = Path('../weights/best.pt').resolve()
TEST_IMG_DIR = Path('../data/dataset/images/test')

# Class order (must match notebook 02)
CLASSES = ['projector', 'whiteboard', 'fire_extinguisher', 'door_sign']

# Evaluation knobs
IMGSZ          = 640
CONF_THRESH    = 0.25
IOU_THRESH     = 0.5
N_QUALITATIVE  = 8
SEED           = 42

random.seed(SEED)
assert WEIGHTS.exists(), 'train the model in notebook 04 first'

## 1. Run validation on the test split

In [ ]:
# ultralytics .val() — overall test-set metrics
model = YOLO(WEIGHTS)
metrics = model.val(data=str(DATA_YAML), split='test', imgsz=IMGSZ,
                    conf=CONF_THRESH, iou=IOU_THRESH)
print('mAP@0.5      :', round(metrics.box.map50, 4))
print('mAP@0.5:0.95 :', round(metrics.box.map,   4))
print('Precision (macro):', round(metrics.box.mp, 4))
print('Recall    (macro):', round(metrics.box.mr, 4))

## 2. Per-class metrics

In [ ]:
# Per-class precision / recall / mAP table
p, r, ap50, ap = metrics.box.p, metrics.box.r, metrics.box.ap50, metrics.box.ap
per_class = pd.DataFrame({
    'class':       CLASSES,
    'precision':   np.round(p, 4),
    'recall':      np.round(r, 4),
    'mAP@0.5':     np.round(ap50, 4),
    'mAP@0.5:0.95': np.round(ap.mean(1) if ap.ndim > 1 else ap, 4),
})
per_class

## 3. Confusion matrix (counts + column-normalized)

In [ ]:
# Plot the confusion matrix Ultralytics computes (last row/col = background)
cm = metrics.confusion_matrix.matrix
labels = CLASSES + ['background']

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cm, annot=True, fmt='.0f', xticklabels=labels, yticklabels=labels, cmap='Blues', ax=ax[0])
ax[0].set_title('Confusion matrix (counts)'); ax[0].set_xlabel('Predicted'); ax[0].set_ylabel('True')

cm_norm = cm / cm.sum(axis=0, keepdims=True).clip(min=1)
sns.heatmap(cm_norm, annot=True, fmt='.2f', xticklabels=labels, yticklabels=labels, cmap='Blues', ax=ax[1])
ax[1].set_title('Confusion matrix (column-normalized)'); ax[1].set_xlabel('Predicted'); ax[1].set_ylabel('True')
plt.tight_layout(); plt.show()

## 4. Per-class PR curves
Ultralytics writes `PR_curve.png` under the val run directory.

In [ ]:
# Display the PR_curve.png produced by .val()
pr_img = Path(metrics.save_dir) / 'PR_curve.png'
if pr_img.exists():
    plt.figure(figsize=(8, 6)); plt.imshow(Image.open(pr_img)); plt.axis('off'); plt.show()
else:
    print('PR_curve.png not found at', pr_img)

## 5. Model-variant comparison
Re-run cells 5–7 after training each variant (notebook 04 → change `CFG['model']`) and fill this table.

In [ ]:
# Placeholder comparison table — overwrite with real numbers from each variant's run
comparison = pd.DataFrame([
    {'variant': 'yolo11n', 'params_M': 2.6,  'mAP@0.5': None, 'mAP@0.5:0.95': None, 'latency_ms': None},
    {'variant': 'yolo11s', 'params_M': 9.4,  'mAP@0.5': round(metrics.box.map50, 4),
                                              'mAP@0.5:0.95': round(metrics.box.map, 4), 'latency_ms': None},
    {'variant': 'yolo11m', 'params_M': 20.0, 'mAP@0.5': None, 'mAP@0.5:0.95': None, 'latency_ms': None},
])
comparison

## 6. Qualitative prediction grid

In [ ]:
# Predict on N_QUALITATIVE random test images and show in a grid
test_imgs = sorted(TEST_IMG_DIR.glob('*.jpg'))
sample = random.sample(test_imgs, min(N_QUALITATIVE, len(test_imgs)))
preds = model.predict(source=[str(p) for p in sample], imgsz=IMGSZ, conf=CONF_THRESH, verbose=False)

cols = 4
rows = (len(preds) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
for ax, r in zip(axes.flatten(), preds):
    ax.imshow(r.plot()[:, :, ::-1])  # BGR -> RGB
    ax.set_title(Path(r.path).name, fontsize=8); ax.axis('off')
for ax in axes.flatten()[len(preds):]:
    ax.axis('off')
plt.tight_layout(); plt.show()

### Reporting checklist
- [ ] Per-class precision / recall / mAP table filled with real numbers
- [ ] Confusion matrix inspected — identify dominant confusion pairs
- [ ] PR curves attached
- [ ] Variant comparison table completed for `yolo11n`, `yolo11s`, `yolo11m`
- [ ] N_QUALITATIVE-image grid included in the report